# Attack Efficiency Analysis — pHash

This notebook mirrors `attack_efficiency_analysis.ipynb` (NeuralHash) exactly,
but targets the four pHash threshold runs:

- `T0.08`, `T0.10`, `T0.12`, `T0.30`

Produces:
- Distance vs Steps (median + IQR)
- Distance vs Time  (median + IQR)
- Success Rate vs Steps
- Success Rate vs Time
- Time-to-success histogram
- Summary stats table
- Threshold sweep comparison table (all 4 thresholds)

**Scientific note on T0.10 vs T0.12:**  
pHash bits change in discrete multiples of 2 (the DCT surrogate flips bit-pairs
simultaneously). The minimum bits needed for T=0.10 is ⌈64×0.10⌉=7, and for
T=0.12 is ⌈64×0.12⌉=8. Because the attack first clears 8 bits (d_norm=0.125),
both thresholds are crossed at the same optimization step. This is a
quantization artifact of the 64-bit hash and is reported as-is.


In [ ]:
import os, sys

# If running from: <repo>/fdeph_eval/analysis
REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))

if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

print("Repo root:", REPO_ROOT)

In [ ]:
import os
import pandas as pd

from fdeph_eval.analysis.plotting import (
    load_attack_csv,
    summarize_run,
    plot_distance_vs_steps,
    plot_distance_vs_time,
    plot_success_rate_vs_steps,
    plot_success_rate_vs_time,
    export_time_to_success_table,
    time_to_success_stats,
    plot_time_to_success_hist,
)

# Primary run: T=0.10 (used for per-image plots, matching nhash notebook convention)
LOG_PATH = os.path.join(REPO_ROOT, "logs", "attack_steps_phash_evasion_mt500_T0.10.csv")
FIG_DIR   = os.path.join(REPO_ROOT, "fdeph_eval", "analysis", "figures")
TABLE_DIR = os.path.join(REPO_ROOT, "fdeph_eval", "analysis", "tables")

os.makedirs(FIG_DIR,   exist_ok=True)
os.makedirs(TABLE_DIR, exist_ok=True)

print("LOG_PATH :", LOG_PATH)
print("FIG_DIR  :", FIG_DIR)
print("TABLE_DIR:", TABLE_DIR)

In [ ]:
df = load_attack_csv(LOG_PATH)
print(f"Rows loaded: {len(df):,}")
df.head()

## Summary statistics (T=0.10)

In [ ]:
summary = summarize_run(df)
summary

## Distance vs Steps

In [ ]:
plot_distance_vs_steps(
    df,
    out_path=f"{FIG_DIR}/phash_distance_vs_steps.png",
    title="pHash Evasion: Distance vs Steps"
)

## Distance vs Time

In [ ]:
plot_distance_vs_time(
    df,
    bin_ms=100,
    out_path=f"{FIG_DIR}/phash_distance_vs_time.png",
    title="pHash Evasion: Distance vs Time"
)

## Success Rate vs Steps

In [ ]:
plot_success_rate_vs_steps(
    df,
    out_path=f"{FIG_DIR}/phash_success_rate_vs_steps.png",
    title="pHash Evasion: Success Rate vs Steps"
)

## Success Rate vs Time

In [ ]:
plot_success_rate_vs_time(
    df,
    bin_ms=100,
    out_path=f"{FIG_DIR}/phash_success_rate_vs_time.png",
    title="pHash Evasion: Success Rate vs Time"
)

## Time-to-success table

In [ ]:
events = export_time_to_success_table(df, f"{TABLE_DIR}/phash_time_to_success.csv")
events.head()

## Time-to-success Histogram

In [ ]:
plot_time_to_success_hist(
    df,
    out_path=f"{FIG_DIR}/phash_time_to_success_hist.png",
    title="pHash Evasion: Time-to-success Histogram"
)

## Per-run summary stats (T=0.10)

In [ ]:
stats = time_to_success_stats(df)
stats

In [ ]:
stats.to_csv(f"{TABLE_DIR}/phash_summary_stats.csv", index=False)
print("Saved:", f"{TABLE_DIR}/phash_summary_stats.csv")

## Threshold sweep analysis (T=0.08 / 0.10 / 0.12 / 0.30)

> **Note:** T=0.10 and T=0.12 produce identical step counts because pHash bits
> flip in discrete jumps. The minimum raw bits for T=0.10 is 7; for T=0.12 is 8.
> The attack first clears 8 bits (d_norm=0.125), satisfying both thresholds
> simultaneously. This is expected and reported as a quantization property of
> 64-bit DCT hashing.

In [ ]:
import pandas as pd
from fdeph_eval.analysis.plotting import load_attack_csv, time_to_success_stats

sweep_paths = {
    "T0.08": os.path.join(REPO_ROOT, "logs", "attack_steps_phash_evasion_mt500_T0.08.csv"),
    "T0.10": os.path.join(REPO_ROOT, "logs", "attack_steps_phash_evasion_mt500_T0.10.csv"),
    "T0.12": os.path.join(REPO_ROOT, "logs", "attack_steps_phash_evasion_mt500_T0.12.csv"),
    "T0.30": os.path.join(REPO_ROOT, "logs", "attack_steps_phash_evasion_mt500_T0.30.csv"),
}

rows = []
for label, p in sweep_paths.items():
    d = load_attack_csv(p)
    r = time_to_success_stats(d)
    r.insert(0, "threshold", label)
    rows.append(r)

sweep = pd.concat(rows, ignore_index=True)
sweep

In [ ]:
sweep.to_csv(f"{TABLE_DIR}/phash_threshold_sweep.csv", index=False)
print("Saved:", f"{TABLE_DIR}/phash_threshold_sweep.csv")

## NeuralHash vs pHash — Direct Comparison Table

Side-by-side at matched thresholds (T=0.08, 0.10, 0.12).  
T=0.30 is pHash-only (NeuralHash not run at this threshold by design —
see project_status.md for rationale).

In [ ]:
nhash_paths = {
    "T0.08": os.path.join(REPO_ROOT, "logs", "attack_steps_nhash_evasion_mt500_T0.08.csv"),
    "T0.10": os.path.join(REPO_ROOT, "logs", "attack_steps_nhash_evasion_mt500_T0.10.csv"),
    "T0.12": os.path.join(REPO_ROOT, "logs", "attack_steps_nhash_evasion_mt500_T0.12.csv"),
}

phash_paths = {
    "T0.08": os.path.join(REPO_ROOT, "logs", "attack_steps_phash_evasion_mt500_T0.08.csv"),
    "T0.10": os.path.join(REPO_ROOT, "logs", "attack_steps_phash_evasion_mt500_T0.10.csv"),
    "T0.12": os.path.join(REPO_ROOT, "logs", "attack_steps_phash_evasion_mt500_T0.12.csv"),
}

comparison_rows = []
for t in ["T0.08", "T0.10", "T0.12"]:
    for method, paths in [("NeuralHash", nhash_paths), ("pHash", phash_paths)]:
        d = load_attack_csv(paths[t])
        r = time_to_success_stats(d)
        r.insert(0, "threshold", t)
        r.insert(1, "hash_method", method)
        comparison_rows.append(r)

comparison = pd.concat(comparison_rows, ignore_index=True)
comparison = comparison.sort_values(["threshold", "hash_method"]).reset_index(drop=True)
comparison

In [ ]:
comparison.to_csv(f"{TABLE_DIR}/nhash_vs_phash_comparison.csv", index=False)
print("Saved:", f"{TABLE_DIR}/nhash_vs_phash_comparison.csv")

## NeuralHash vs pHash — Overlay Plot (Success Rate vs Steps)

Both hashes on the same axes at T=0.10, showing the dramatic speed difference.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np

def success_rate_curve_steps(df):
    """Return (steps, cumulative_success_rate) arrays."""
    success_df = df[df['success'] == 1].copy()
    success_df = success_df.sort_values('step')
    n_total = df['image_id'].nunique()
    steps = success_df['step'].values
    cum_rate = np.arange(1, len(steps) + 1) / n_total
    return steps, cum_rate

nhash_df = load_attack_csv(
    os.path.join(REPO_ROOT, "logs", "attack_steps_nhash_evasion_mt500_T0.10.csv")
)
phash_df = load_attack_csv(
    os.path.join(REPO_ROOT, "logs", "attack_steps_phash_evasion_mt500_T0.10.csv")
)

nx, ny = success_rate_curve_steps(nhash_df)
px, py = success_rate_curve_steps(phash_df)

fig, ax = plt.subplots(figsize=(9, 5))
ax.step(nx, ny, where='post', color='#1f77b4', linewidth=1.8, label='NeuralHash')
ax.step(px, py, where='post', color='#d62728', linewidth=1.8, label='pHash')
ax.set_xlabel('Step', fontsize=12)
ax.set_ylabel('Success rate', fontsize=12)
ax.set_title('NeuralHash vs pHash: Success Rate vs Steps (T=0.10)', fontsize=13)
ax.legend(fontsize=11)
ax.set_ylim(0, 1.05)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
out = f"{FIG_DIR}/nhash_vs_phash_success_rate_vs_steps.png"
plt.savefig(out, dpi=150, bbox_inches='tight')
plt.show()
print("Saved:", out)

## NeuralHash vs pHash — Overlay Plot (Success Rate vs Time)

In [ ]:
def success_rate_curve_time(df, bin_ms=250):
    """Return (time_bins, cumulative_success_rate) arrays."""
    success_df = df[df['success'] == 1].copy()
    success_df = success_df.sort_values('elapsed_ms')
    n_total = df['image_id'].nunique()
    times = success_df['elapsed_ms'].values
    cum_rate = np.arange(1, len(times) + 1) / n_total
    return times, cum_rate

nt, nr = success_rate_curve_time(nhash_df)
pt, pr = success_rate_curve_time(phash_df)

fig, ax = plt.subplots(figsize=(9, 5))
ax.step(nt, nr, where='post', color='#1f77b4', linewidth=1.8, label='NeuralHash')
ax.step(pt, pr, where='post', color='#d62728', linewidth=1.8, label='pHash')
ax.set_xlabel('Elapsed time (ms)', fontsize=12)
ax.set_ylabel('Success rate', fontsize=12)
ax.set_title('NeuralHash vs pHash: Success Rate vs Time (T=0.10)', fontsize=13)
ax.legend(fontsize=11)
ax.set_ylim(0, 1.05)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
out = f"{FIG_DIR}/nhash_vs_phash_success_rate_vs_time.png"
plt.savefig(out, dpi=150, bbox_inches='tight')
plt.show()
print("Saved:", out)

## pHash Threshold Sweep — Steps Bar Chart

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))

thresholds = ["T0.08", "T0.10", "T0.12", "T0.30"]
medians_steps = sweep.set_index('threshold')['median_steps'].reindex(thresholds)
p95_steps     = sweep.set_index('threshold')['p95_steps'].reindex(thresholds)
medians_time  = sweep.set_index('threshold')['median_time_ms'].reindex(thresholds)
p95_time      = sweep.set_index('threshold')['p95_time_ms'].reindex(thresholds)

x = np.arange(len(thresholds))
w = 0.35

# Steps
axes[0].bar(x - w/2, medians_steps, w, label='Median', color='#1f77b4')
axes[0].bar(x + w/2, p95_steps,     w, label='P95',    color='#aec7e8')
axes[0].set_xticks(x)
axes[0].set_xticklabels(thresholds)
axes[0].set_ylabel('Optimization steps')
axes[0].set_title('pHash: Steps to Success by Threshold')
axes[0].legend()
axes[0].grid(axis='y', alpha=0.3)

# Time
axes[1].bar(x - w/2, medians_time / 1000, w, label='Median', color='#d62728')
axes[1].bar(x + w/2, p95_time     / 1000, w, label='P95',    color='#f7b6b6')
axes[1].set_xticks(x)
axes[1].set_xticklabels(thresholds)
axes[1].set_ylabel('Time (seconds)')
axes[1].set_title('pHash: Time to Success by Threshold')
axes[1].legend()
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
out = f"{FIG_DIR}/phash_threshold_sweep_bars.png"
plt.savefig(out, dpi=150, bbox_inches='tight')
plt.show()
print("Saved:", out)

## Print key numbers for thesis write-up

In [ ]:
print("=" * 60)
print("KEY NUMBERS FOR THESIS — pHash Evasion")
print("=" * 60)
print("\npHash threshold sweep (500 images each):")
print(sweep[['threshold','n_images','n_succeeded','success_rate',
             'median_steps','p95_steps','median_time_ms','p95_time_ms']].to_string(index=False))

print("\n\nNeuralHash vs pHash at T=0.10:")
row = comparison[comparison['threshold']=='T0.10'][['hash_method','median_steps',
                 'p95_steps','median_time_ms','p95_time_ms']]
print(row.to_string(index=False))

print("\n\nSpeedup factor (NeuralHash median steps / pHash median steps at T=0.10):")
nhash_med = comparison[
    (comparison['threshold']=='T0.10') & (comparison['hash_method']=='NeuralHash')
]['median_steps'].values[0]
phash_med = comparison[
    (comparison['threshold']=='T0.10') & (comparison['hash_method']=='pHash')
]['median_steps'].values[0]
print(f"  NeuralHash median steps : {nhash_med:.0f}")
print(f"  pHash      median steps : {phash_med:.0f}")
print(f"  Speedup                 : {nhash_med/phash_med:.1f}×  (pHash is faster to evade)")